### Step 1: Install Dependencies

In [1]:
# Install the necessary libraries for our pipeline
#!pip install -q duckdb duckdb-engine langchain langchain-classic langchain-community langchain-ollama sqlalchemy==2.0.44

In [2]:
import duckdb
import os
import pandas as pd

# File path
db_path = "apple_sales_rag_ollama.db"
csv_path = "../data/processed/cleaned_apple_sales_v3.csv"

try:
    duckdb.close()  
except: pass
if os.path.exists(db_path):
    try: os.remove(db_path)
    except OSError:
        
        import time
        db_path = f"apple_sales_rag_{int(time.time())}.db"
        print(f"\u26a0 Previous DB locked, using: {db_path}")

print(f"Connecting to DuckDB and loading dataset from {csv_path}...")
con = duckdb.connect(db_path)
con.execute(f"CREATE TABLE sales AS SELECT * FROM read_csv_auto('{csv_path}')")
print("\u2705 Successfully loaded 1 Million rows into DuckDB!")

print("\nSchema (What the Local LLM sees):")
display(con.execute("DESCRIBE sales").df())
con.close()


Connecting to DuckDB and loading dataset from ../data/processed/cleaned_apple_sales_v3.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Successfully loaded 1 Million rows into DuckDB!

Schema (What the Local LLM sees):


,column_name,column_type,null,key,default,extra
0,sale_id,VARCHAR,YES,None,None,None
1,sale_date,DATE,YES,None,None,None
2,store_id,VARCHAR,YES,None,None,None
3,product_id,VARCHAR,YES,None,None,None
4,quantity,BIGINT,YES,None,None,None
5,product_name,VARCHAR,YES,None,None,None
6,launch_date,DATE,YES,None,None,None
7,price,DOUBLE,YES,None,None,None
8,store_name,VARCHAR,YES,None,None,None
9,city,VARCHAR,YES,None,None,None


In [3]:
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_ollama import ChatOllama
from langchain_community.tools import QuerySQLDatabaseTool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Connect LangChain to DuckDB
custom_schema = """
CREATE TABLE sales (
    sale_date TIMESTAMP,
    store_name VARCHAR,
    city VARCHAR,
    country_norm_mapped VARCHAR,
    product_name VARCHAR,
    category_name VARCHAR,
    sales_amount_realistic DOUBLE,
    quantity_realistic DOUBLE,
    price_realistic DOUBLE,
    year BIGINT,
    month BIGINT
);
"""
db = SQLDatabase.from_uri(f"duckdb:///{db_path}", custom_table_info={"sales": custom_schema})


# 2. Initialize Qwen2.5 (temperature=0 for SQL, slightly warm for analysis)
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0)
analyst_llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.0)

# 3. SQL Generation Prompt — 15 strict business rules for accurate query generation
custom_prompt = PromptTemplate.from_template(
    """You are an elite DuckDB SQL programming assistant answering questions about Apple Retail Sales data.
    Given an input question, create a syntactically correct DuckDB query to run.
    
    Never query for all the columns from a specific table, only ask for the few relevant columns given the question.
    Be careful to not query for columns that do not exist.
    
    CRITICAL APPLE RETAIL BUSINESS RULES:
    1. If asked about "Sales", "Revenue", or "Income", use the 'sales_amount_realistic' column.
    2. If asked about Volume or Item Counts, use 'quantity_realistic'.
    3. If asked about a Country, filter using 'country_norm_mapped'.
    4. There is ONLY ONE table named 'sales'. DO NOT JOIN other tables.
    5. If asked to count "transactions" or "orders", use COUNT(*).
    6. If comparing metrics across different years, use conditional aggregation: SUM(CASE WHEN year=2023 THEN column END) AS year_2023, SUM(CASE WHEN year=2024 THEN column END) AS year_2024.
    7. Output ONLY the raw SQL string. No markdown, no explanation.
    8. To find the "most" or "least", use ORDER BY + LIMIT 1, never MAX()/MIN() with unaggregated columns.
    9. When asked for the PRICE of a product, ALWAYS use 'price_realistic'. NEVER use 'sales_amount_realistic' for prices.
    10. When filtering for a specific product line, use product_name ILIKE '%ProductName%'.
    
    ADDITIONAL CRITICAL RULES:
    11. When asked about GDP or GDP per capita, ALWAYS use the 'gdp_per_capita' column. GDP is NOT revenue.
    12. IMPORTANT PRODUCT MATCHING: If a user asks for a base model like 'iPhone 13' or 'iPhone 14', they mean ONLY the base model. You MUST use exact matching: `product_name ILIKE 'iPhone 13'`. Do NOT use `%` wildcards (`ILIKE '%iPhone 13%'`) because that will accidentally include 'Pro', 'mini', and 'Plus'. ONLY use wildcards if they explicitly ask for the 'iPhone 13 family' or 'all iPhone 13s'.
    13. When asked 'which products can I afford' or about a budget, ALWAYS: (a) use AVG(price_realistic) grouped by product_name, (b) filter with HAVING AVG(price_realistic) <= budget, (c) ONLY include the product category the user asked about (e.g. if they ask about iPhones, add WHERE product_name ILIKE '%iPhone%').
    14. When comparing exactly 2 cities or locations, ALWAYS add WHERE city IN ('City1', 'City2') to filter only those cities. YOU MUST ALSO include the 'city' column in the SELECT and GROUP BY clauses.
    15. When asked if a price 'dropped' or 'changed' between years, use conditional aggregation to get separate values per year, e.g.: AVG(CASE WHEN year=2023 THEN price_realistic END) AS price_2023, AVG(CASE WHEN year=2024 THEN price_realistic END) AS price_2024.
    16. VERY IMPORTANT: ALL string values in 'country_norm_mapped' are STRICTLY LOWERCASE (e.g., 'united states'). Always use exact lowercase strings when filtering by country! Also use ILIKE for product_name to be case-insensitive.
    17. ALWAYS SELECT the columns you are GROUPING BY. If you GROUP BY year, you MUST include year in the SELECT clause.
    18. TIME SERIES GROUPING: If the user asks 'for each year' or 'by year', you MUST `GROUP BY year` and `SELECT year`. If they ask 'by month', you MUST `GROUP BY month` and `SELECT month`. Do NOT add any extra grouping columns (like city, country, or store) unless explicitly requested.

    EXAMPLES:
    User: "What was total revenue in 2024?"
    SQL: SELECT SUM(sales_amount_realistic) FROM sales WHERE year = 2024;
    
    User: "What is the average price of iPhone 13 and iPhone 14 for each year?"
    SQL: SELECT product_name, year, AVG(price_realistic) FROM sales WHERE (product_name ILIKE 'iPhone 13' OR product_name ILIKE 'iPhone 14') GROUP BY product_name, year;
    
    User: "Which country had the most transactions?"
    SQL: SELECT country_norm_mapped, COUNT(*) FROM sales GROUP BY country_norm_mapped ORDER BY COUNT(*) DESC LIMIT 1;
    
    User: "Compare the average price of the iPhone 14 between London and New York"
    SQL: SELECT city, AVG(price_realistic) FROM sales WHERE product_name ILIKE 'iPhone 14' AND city IN ('London', 'New York') GROUP BY city;
    
    User: "Did the price of MacBook Air change between 2023 and 2024?"
    SQL: SELECT product_name, AVG(CASE WHEN year=2023 THEN price_realistic END) AS price_2023, AVG(CASE WHEN year=2024 THEN price_realistic END) AS price_2024 FROM sales WHERE product_name ILIKE '%MacBook Air%' GROUP BY product_name;
    
    Only use the following tables:
    {table_info}

    Return a maximum of {top_k} results unless otherwise specified.

    Question: {input}"""
)

# 4. CEO Analyst Assistant Prompt — converts raw SQL results into executive insights
analyst_prompt = ChatPromptTemplate.from_template(
    """You are a Senior Data Analyst presenting findings to Apple's executive leadership.

Your communication style:
- Professional, confident, and concise
- Lead with the key insight first, then supporting details
- Use exact numbers with proper formatting (commas, currency symbols, percentages)
- Add brief business context or actionable takeaways when relevant
- If the data shows a trend, highlight it
- Keep responses to 2-4 sentences for simple queries, up to a short paragraph for complex ones
- Never mention SQL, databases, tables, columns, or technical implementation details

CRITICAL ANTI-HALLUCINATION RULES:
1. You MUST rely EXCLUSIVELY on the data provided in 'The data returned'. Do NOT use your pre-trained knowledge to fill in prices, dates, or sales figures.
2. If 'The data returned' is empty (e.g., `[]`, `""`, or `None`), you MUST explicitly state that there is no data available for this query. Do NOT invent a number. 
3. For example, if asked about a product in a year before it launched, the data will be empty. Explain that the product likely did not exist or had no sales in that period.
4. Do NOT perform mathematical calculations (like averaging multiple product models together). Only report the exact numbers and exact product names provided in the SQL result.

The user asked: "{question}"

The data returned: {result}

Provide your executive briefing:"""
)

# 5. Build the chains
write_query = create_sql_query_chain(llm, db, prompt=custom_prompt)
execute_query = QuerySQLDatabaseTool(db=db)
analyst_chain = analyst_prompt | analyst_llm | StrOutputParser()

print("\u2705 SQL Engine + CEO Analyst Assistant are ready!")
print("   \u2699  SQL Generation: Qwen 2.5 (temperature=0, 15 business rules)")
print("   \U0001f4ca Analyst Persona: Qwen 2.5 (temperature=0.0)")


✅ SQL Engine + CEO Analyst Assistant are ready!
   ⚙  SQL Generation: Qwen 2.5 (temperature=0, 15 business rules)
   📊 Analyst Persona: Qwen 2.5 (temperature=0.0)


d:\anaconda\envs\Apple\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


In [4]:
def ask_local_ai(question):
    """Ask the local AI a question and get an executive-style briefing."""
    print(f"\n{'='*60}")
    print(f"\U0001f4ac  {question}")
    print(f"{'='*60}")
    
    # Step 1: Generate SQL
    sql_query = write_query.invoke({"question": question})
    clean_sql = sql_query.replace("```sql", "").replace("```", "").replace("SQLQuery:", "").strip()
    
    # Safety: inject LIMIT if not present
    if "LIMIT" not in clean_sql.upper():
        clean_sql = clean_sql.rstrip(";") + " LIMIT 50;"
    
    print(f"\n\u2699  SQL: {clean_sql}")
    
    # Step 2: Execute SQL
    try:
        raw_result = execute_query.invoke(clean_sql)
    except Exception as e:
        print(f"\n\u274c  Query failed: {e}")
        return
    
    # Step 3: Generate executive briefing via analyst LLM
    print(f"\n\U0001f50d  Raw Data: {str(raw_result)[:200]}{'...' if len(str(raw_result)) > 200 else ''}")
    
    try:
        briefing = analyst_chain.invoke({
            "question": question,
            "result": raw_result
        })
        print(f"\n\U0001f4ca  Analyst Briefing:")
        print(f"   {briefing}")
    except Exception as e:
        # Fallback to raw result if analyst LLM fails
        print(f"\n\U0001f4cb  Result: {raw_result}")
    
    print()


#### Easy Level Tests (Standard Analytics)

In [5]:
ask_local_ai("How many unique stores do we have in our entire dataset?")
ask_local_ai("What are the distinct product categories we sell? List them out.")
ask_local_ai("Which country had the highest number of overall sales transactions (not volume, just number of rows)?")


💬  How many unique stores do we have in our entire dataset?

⚙  SQL: SELECT COUNT(DISTINCT store_name) FROM sales LIMIT 50;

🔍  Raw Data: [(69,)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** We have 69 unique stores across our entire dataset.

**Supporting Details:** This figure represents the total number of distinct locations where sales transactions occurred. Each store is counted once, regardless of how many products were sold there or the frequency of visits.

**Business Context:** Understanding the number of unique stores helps in planning inventory management, resource allocation, and market expansion strategies. With 69 stores, we can tailor our marketing efforts to specific regions more effectively.

**Actionable Takeaways:**
- **Inventory Management:** Ensure that inventory levels are optimized for each store to maximize sales.
- **Marketing Strategy:** Develop targeted marketing campaigns for the top-performing stores to drive higher sales and market sh

#### Medium Level Tests (Mathematical Inference & Data rules)

In [6]:
ask_local_ai("Which country sold the absolute most physical items (volume) out of all the countries combined?")
ask_local_ai("What is the average Apple revenue for the iPhone 14 in Japan in 2024? Remember to use the realistic amount.")



💬  Which country sold the absolute most physical items (volume) out of all the countries combined?

⚙  SQL: SELECT country_norm_mapped, SUM(quantity_realistic) AS total_volume_sold
FROM sales
GROUP BY country_norm_mapped
ORDER BY total_volume_sold DESC
LIMIT 1;

🔍  Raw Data: [('united states', 1663887)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** The United States sold the absolute most physical items (volume) out of all countries combined, with a total volume of **1,663,887 units**.

**Business Context:** This significant sales figure underscores the United States' strong market position in the global retail industry. It highlights the country's robust consumer base and its ability to attract and retain customers across various product categories.

**Actionable Takeaways:**

- **Strategic Focus:** Apple should continue to prioritize expanding its presence in the United States, leveraging its established customer base and competitive pricing strategies.
  
- **Ma

#### Advanced Level Tests (HAVING clauses & Time Series Grouping)

In [7]:
ask_local_ai("top 5 stores by sales in years")


💬  top 5 stores by sales in years

⚙  SQL: SELECT store_name, SUM(sales_amount_realistic) AS total_sales
FROM sales
GROUP BY store_name
ORDER BY total_sales DESC
LIMIT 5;

🔍  Raw Data: [('Apple Central World', 154258641.4197363), ('Apple Covent Garden', 153205281.50775176), ('Apple Champs-Elysees', 134424367.88117734), ('Apple The Dubai Mall', 117578827.29149304), ('Apple Orchard Ro...

📊  Analyst Briefing:
   **Executive Briefing**

The top 5 stores by sales in the past year are:

1. **Apple Central World**: $15,425,864,119.74
2. **Apple Covent Garden**: $15,320,528,150.78
3. **Apple Champs-Elysees**: $13,442,436,788.12
4. **Apple The Dubai Mall**: $11,757,882,729.15
5. **Apple Orchard Road**: $10,661,229,289.31

**Actionable Takeaways:**

- Apple Central World and Covent Garden continue to dominate the sales market with significant margins.
- Apple Champs-Elysees has shown steady growth, indicating a strong presence in Paris.
- The Dubai Mall demonstrates robust sales performance, s

In [8]:
ask_local_ai("top 10 products")


💬  top 10 products

⚙  SQL: SELECT product_name, SUM(sales_amount_realistic) AS total_sales FROM sales GROUP BY product_name ORDER BY total_sales DESC LIMIT 10;

🔍  Raw Data: [('iPhone 17 Pro', 199146894.96784645), ('iPhone 17 Pro Max', 186202218.4274062), ('iPhone 16 Plus', 182509576.8192973), ('iPhone 16 Pro Max', 178272580.90143302), ('iPhone 17', 160964590.07859108), (...

📊  Analyst Briefing:
   **Executive Briefing**

The top 10 products based on sales are:

1. **iPhone 17 Pro**: Sold for $19,914,689,4.97
2. **iPhone 17 Pro Max**: Sold for $18,620,221,8.43
3. **iPhone 16 Plus**: Sold for $18,250,957,6.82
4. **iPhone 16 Pro Max**: Sold for $17,827,258,0.90
5. **iPhone 17**: Sold for $16,096,459,0.08
6. **iPhone 16 Pro**: Sold for $15,403,783,8.46
7. **iPhone 15 Plus**: Sold for $13,866,658,9.06
8. **iPhone 16**: Sold for $13,780,314,7.22
9. **iPhone 15 Pro**: Sold for $13,622,399,3.69
10. **Mac Pro (M2 Ultra)**: Sold for $13,037,755,6.89

**Actionable Takeaways:**

- The iPhone 1

In [9]:
ask_local_ai("top city in months")


💬  top city in months

⚙  SQL: SELECT city, COUNT(*) AS transaction_count
FROM sales
GROUP BY city, month
ORDER BY transaction_count DESC
LIMIT 1;

🔍  Raw Data: [('Paris', 4961)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** Paris is currently the top city in terms of sales, with a total of 4,961 units sold.

**Supporting Details:** This figure represents the highest monthly sales volume recorded for any city in our dataset. The data shows that Paris consistently outperforms other cities throughout the year, indicating its strong market presence and customer base.

**Business Context:** Understanding which city performs best can help Apple tailor marketing strategies to maximize sales and customer engagement. By focusing on Paris, Apple can leverage its existing customer base and potential for further growth in this high-performing market.

**Actionable Takeaways:**
- **Marketing Strategy:** Consider enhancing marketing efforts in Paris to capitalize on its strong 

In [10]:
ask_local_ai("how many columns do we have not from sales")


💬  how many columns do we have not from sales

⚙  SQL: SELECT count(*) FROM information_schema.columns WHERE table_name = 'sales' LIMIT 50;

🔍  Raw Data: [(35,)]

📊  Analyst Briefing:
   We have 35 columns available.



In [11]:
ask_local_ai("how many rows do we have")


💬  how many rows do we have

⚙  SQL: SELECT COUNT(*) FROM sales LIMIT 50;

🔍  Raw Data: [(1068918,)]

📊  Analyst Briefing:
   We have 1,068,918 rows of data available. This indicates a substantial dataset that covers various aspects of our business operations and customer interactions.



In [12]:
ask_local_ai("What is the average price of the iPhone 13 and iPhone 14 by year? Filter using product LIKE '%iPhone 13%' OR product LIKE '%iPhone 14%'. Do NOT include other iPhones.")



💬  What is the average price of the iPhone 13 and iPhone 14 by year? Filter using product LIKE '%iPhone 13%' OR product LIKE '%iPhone 14%'. Do NOT include other iPhones.

⚙  SQL: SELECT product_name, year, AVG(price_realistic) AS average_price
FROM sales
WHERE (product_name ILIKE 'iPhone 13' OR product_name ILIKE 'iPhone 14')
GROUP BY product_name, year LIMIT 50;

🔍  Raw Data: [('iPhone 13', 2024, 646.7346167649283), ('iPhone 14', 2024, 704.2181905774314), ('iPhone 13', 2021, 791.6215596360211), ('iPhone 14', 2025, 632.9231111001288), ('iPhone 13', 2025, 630.8125269478052),...

📊  Analyst Briefing:
   The average price of the iPhone 13 and iPhone 14 by year is as follows:

- **iPhone 13**: 
  - 2021: $791.62
  - 2022: $756.05
  - 2023: $718.29
  - 2024: $646.73
  - 2025: $630.81

- **iPhone 14**: 
  - 2022: $806.26
  - 2023: $781.28
  - 2024: $704.22
  - 2025: $632.92

The iPhone 14 has shown a slight increase in average price compared to the iPhone 13 over the years, with the highest

In [13]:
ask_local_ai("What is the average price of the iPhone 13 and iPhone 14 by year?")


💬  What is the average price of the iPhone 13 and iPhone 14 by year?

⚙  SQL: SELECT product_name, year, AVG(price_realistic) AS avg_price
FROM sales
WHERE (product_name ILIKE 'iPhone 13' OR product_name ILIKE 'iPhone 14')
GROUP BY product_name, year LIMIT 50;

🔍  Raw Data: [('iPhone 13', 2023, 718.2856305294363), ('iPhone 14', 2023, 781.280033637729), ('iPhone 13', 2024, 646.7346167649283), ('iPhone 14', 2024, 704.2181905774314), ('iPhone 13', 2021, 791.6215596360211), ...

📊  Analyst Briefing:
   The average price of the iPhone 13 by year is $718.29, and for the iPhone 14, it is $781.28.

**Actionable Takeaway:** Apple should consider offering a more affordable option for the iPhone 13 in 2025 to attract budget-conscious consumers who may have been deterred by its higher price compared to the iPhone 14.



In [14]:
ask_local_ai("What is the average price of the iPhone 13 and iPhone 14 by years?Do NOT include other iPhones")


💬  What is the average price of the iPhone 13 and iPhone 14 by years?Do NOT include other iPhones

⚙  SQL: SELECT product_name, year, AVG(price_realistic) AS avg_price
FROM sales
WHERE (product_name ILIKE 'iPhone 13' OR product_name ILIKE 'iPhone 14')
GROUP BY product_name, year LIMIT 50;

🔍  Raw Data: [('iPhone 14', 2023, 781.280033637729), ('iPhone 13', 2023, 718.2856305294363), ('iPhone 14', 2024, 704.2181905774314), ('iPhone 13', 2024, 646.7346167649283), ('iPhone 14', 2025, 632.9231111001288), ...

📊  Analyst Briefing:
   The average price of the iPhone 13 and iPhone 14 by years is as follows:

- **iPhone 13**: 
  - 2023: $718.29
  - 2024: $646.73
  - 2025: $630.81

- **iPhone 14**: 
  - 2023: $781.28
  - 2024: $704.22
  - 2025: $632.92

The iPhone 14 has a higher average price compared to the iPhone 13 across all years, with an increase of approximately $63.08 from 2023 to 2025. This trend suggests that the iPhone 14 may be more expensive than the iPhone 13 in recent years.



In [15]:
ask_local_ai("Which iPhone models Show product and AVG(price_realistic), filter WHERE product LIKE '%iPhone%', GROUP BY product, HAVING AVG(price_realistic) <= 1000, ORDER BY AVG(price_realistic).")



💬  Which iPhone models Show product and AVG(price_realistic), filter WHERE product LIKE '%iPhone%', GROUP BY product, HAVING AVG(price_realistic) <= 1000, ORDER BY AVG(price_realistic).

⚙  SQL: SELECT product_name, AVG(price_realistic) 
FROM sales 
WHERE product_name ILIKE '%iPhone%' 
GROUP BY product_name 
HAVING AVG(price_realistic) <= 1000 
ORDER BY AVG(price_realistic) LIMIT 50;

🔍  Raw Data: [('iPhone SE (2nd Gen)', 356.834641027497), ('iPhone SE (3rd Gen)', 422.14109421217506), ('iPhone 11', 576.9902469318575), ('iPhone 16e', 603.6950065943845), ('iPhone 12 mini', 624.3616718220374), ('i...

📊  Analyst Briefing:
   **Executive Briefing**

The data shows that the iPhone SE (2nd Gen) and iPhone SE (3rd Gen) have the lowest average realistic prices, both below $1,000. The iPhone 11 Pro Max has the highest average realistic price at approximately $909. This indicates a significant difference in pricing strategies for different models of iPhones.

**Actionable Takeaways:**

- **Pric

In [16]:
ask_local_ai("Which iPhone models Show products and AVG price_realistic")


💬  Which iPhone models Show products and AVG price_realistic

⚙  SQL: SELECT product_name, AVG(price_realistic) AS avg_price_realistic FROM sales WHERE product_name ILIKE '%iPhone%' GROUP BY product_name LIMIT 50;

🔍  Raw Data: [('iPhone 13 mini', 674.7017382541678), ('iPhone 12 Pro Max', 980.8778085717026), ('iPhone 17 Pro', 1007.4443325472959), ('iPhone 14 Plus', 885.5952315256492), ('iPhone 12 mini', 624.3616718220375), (...

📊  Analyst Briefing:
   The data returned shows that the iPhone models with products and an average realistic price are:

- iPhone 13 mini: $674.70
- iPhone 12 Pro Max: $980.88
- iPhone 17 Pro: $1,007.44
- iPhone 14 Plus: $885.60
- iPhone SE (3rd Gen): $422.14

These models have an average realistic price of approximately $903.11.



In [17]:
ask_local_ai("Which iPhone models have an average price_realistic under $1000?") 


💬  Which iPhone models have an average price_realistic under $1000?

⚙  SQL: SELECT product_name, AVG(price_realistic) AS avg_price
FROM sales
WHERE category_name = 'smartphones' AND product_name ILIKE '%iPhone%'
GROUP BY product_name
HAVING AVG(price_realistic) <= 1000 LIMIT 50;

🔍  Raw Data: 

📊  Analyst Briefing:
   Based on the data provided, there are no iPhone models with an average price_realistic under $1000. The lowest-priced model listed is the iPhone 8, which costs $749. This indicates that all iPhones currently available have a price_realistic of at least $749, making it impossible for any to meet the criteria of having an average price_realistic under $1000.



In [18]:
# Affordability Query — Rule 13 ensures aggregated price comparison
ask_local_ai("What is the average price_realistic of the iPhone 14 and the average price_realistic of any iPad? Show product and AVG(price_realistic). Filter WHERE product LIKE '%iPhone 14%' OR product LIKE '%iPad%'. GROUP BY product.")



💬  What is the average price_realistic of the iPhone 14 and the average price_realistic of any iPad? Show product and AVG(price_realistic). Filter WHERE product LIKE '%iPhone 14%' OR product LIKE '%iPad%'. GROUP BY product.

⚙  SQL: SELECT product_name, AVG(price_realistic) 
FROM sales 
WHERE (product_name ILIKE '%iPhone 14%' OR product_name ILIKE '%iPad%') 
GROUP BY product_name LIMIT 50;

🔍  Raw Data: [('iPad mini (6th Gen)', 481.60910638226613), ('iPad Air (3rd Gen)', 411.47386075300705), ('iPad Pro 12.9-inch (4th Gen)', 893.7321261104629), ('iPad Air (M2)', 602.3127205871326), ('iPad Pro 11-inch ...

📊  Analyst Briefing:
   **Executive Briefing**

The average price_realistic for the iPhone 14 is **$786.25**, and the average price_realistic for any iPad is **$602.31**.

### Key Insights:
- The iPhone 14 has a higher average price than any iPad, indicating it may be more expensive or in higher demand.
- The iPad Air (M2) has the lowest average price among all iPads listed, suggestin

In [19]:
# City Price Comparison — Rule 14 ensures only 2 cities are compared
ask_local_ai("Compare the average price_realistic of the iPhone 14 between London and New York. Filter WHERE product LIKE '%iPhone 14%' AND city IN ('London', 'New York'). GROUP BY city.")
ask_local_ai("What was the most expensive item sold at the 'Apple Covent Garden' store? Show product, price_realistic. Filter WHERE store_name = 'Apple Covent Garden'. ORDER BY price_realistic DESC LIMIT 1.")



💬  Compare the average price_realistic of the iPhone 14 between London and New York. Filter WHERE product LIKE '%iPhone 14%' AND city IN ('London', 'New York'). GROUP BY city.

⚙  SQL: SELECT city, AVG(price_realistic) AS avg_price 
FROM sales 
WHERE product_name ILIKE '%iPhone 14%' AND city IN ('London', 'New York') 
GROUP BY city LIMIT 50;

🔍  Raw Data: [('New York', 951.4505082835534), ('London', 937.5550820231606)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** The average price_realistic of the iPhone 14 in New York is $951.45, while in London it is $937.56.

**Supporting Details:**
- **New York:** The average price for iPhones 14 sold in New York was $951.45.
- **London:** The average price for iPhones 14 sold in London was $937.56.

**Business Context:** This comparison highlights the slight difference in pricing between the two major cities, which could be influenced by factors such as local demand, supply chain dynamics, or market conditions specific to eac

In [20]:
ask_local_ai("Compare the average price_realistic of the iPhone 14 between London and New York")


💬  Compare the average price_realistic of the iPhone 14 between London and New York

⚙  SQL: SELECT city, AVG(price_realistic) AS avg_price 
FROM sales 
WHERE product_name ILIKE 'iPhone 14' AND city IN ('London', 'New York') 
GROUP BY city LIMIT 50;

🔍  Raw Data: [('New York', 798.5712216961501), ('London', 794.0329852679114)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** The average price_realistic of the iPhone 14 in New York is $798.57, while in London it is $794.03.

**Supporting Details:**
- **New York:** The highest average price_realistic for the iPhone 14 among the cities provided.
- **London:** The second-highest average price_realistic, indicating a slight premium over New York.

**Business Context:** This comparison highlights regional pricing dynamics and could inform Apple's marketing strategies or pricing adjustments in different markets. For instance, if Apple aims to increase sales in London, it might consider offering promotions or discounts to att

In [21]:
ask_local_ai("Are there any MacBooks available that have an average price under $1500 in 2024?")


💬  Are there any MacBooks available that have an average price under $1500 in 2024?

⚙  SQL: SELECT product_name, AVG(price_realistic) AS avg_price_2024 
FROM sales 
WHERE category_name = 'laptops' AND year = 2024 
GROUP BY product_name 
HAVING AVG(price_realistic) <= 1500 LIMIT 50;

🔍  Raw Data: 

📊  Analyst Briefing:
   Yes, there are MacBook models available with an average price under $1500 in 2024. The most affordable model is the Apple MacBook Air (M1), which has an average price of $999. This model offers a powerful and lightweight laptop that meets the budget requirement.



In [22]:
ask_local_ai("Are there any iphone available that have an average price under $700 in 2024?")


💬  Are there any iphone available that have an average price under $700 in 2024?

⚙  SQL: SELECT product_name, AVG(price_realistic) AS avg_price_2024 
FROM sales 
WHERE year = 2024 AND product_name ILIKE '%iPhone%' 
GROUP BY product_name 
HAVING AVG(price_realistic) <= 700 LIMIT 50;

🔍  Raw Data: [('iPhone 13 mini', 561.335675384428), ('iPhone SE (3rd Gen)', 379.90315526033896), ('iPhone 12 mini', 560.9680690913729), ('iPhone 12', 642.6970031482907), ('iPhone 13', 646.7346167649283), ('iPhone ...

📊  Analyst Briefing:
   Yes, there are several iPhone models available with an average price under $700 in 2024. The iPhone 13 mini and iPhone SE (3rd Gen) have prices of approximately $561 and $380 respectively, which fall below the specified threshold.



In [23]:
ask_local_ai("What is the absolute cheapest product I can buy from the 'Accessories' category?")


💬  What is the absolute cheapest product I can buy from the 'Accessories' category?

⚙  SQL: SELECT product_name, MIN(price_realistic) AS lowest_price
FROM sales
WHERE category_name = 'Accessories'
GROUP BY product_name LIMIT 50;

🔍  Raw Data: [('AirTag', 19.021486822024002), ('Magic Keyboard for iPad Pro', 194.26090424005253)]

📊  Analyst Briefing:
   The absolute cheapest product in the 'Accessories' category is the AirTag, priced at $19.02. This makes it an excellent choice for those looking to save money on essential accessories.



In [24]:
# Year-over-Year Price Comparison — Rule 15 ensures conditional aggregation
ask_local_ai("Compare the average price_realistic of the iPhone 13 in 2023 vs 2024. Use: AVG(CASE WHEN year=2023 THEN price_realistic END) AS price_2023, AVG(CASE WHEN year=2024 THEN price_realistic END) AS price_2024. Filter WHERE product LIKE '%iPhone 13%' AND year IN (2023, 2024).")



💬  Compare the average price_realistic of the iPhone 13 in 2023 vs 2024. Use: AVG(CASE WHEN year=2023 THEN price_realistic END) AS price_2023, AVG(CASE WHEN year=2024 THEN price_realistic END) AS price_2024. Filter WHERE product LIKE '%iPhone 13%' AND year IN (2023, 2024).

⚙  SQL: SELECT 
    AVG(CASE WHEN year = 2023 THEN price_realistic END) AS price_2023,
    AVG(CASE WHEN year = 2024 THEN price_realistic END) AS price_2024
FROM 
    sales
WHERE 
    product_name ILIKE '%iPhone 13%' AND year IN (2023, 2024) LIMIT 50;

🔍  Raw Data: [(805.5495061122996, 718.275062489882)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** The average price_realistic of the iPhone 13 has decreased from $805.55 in 2023 to $718.28 in 2024.

**Supporting Details:**
- **Price in 2023:** $805.55
- **Price in 2024:** $718.28

**Business Context:** This trend suggests a slight decline in the average price of the iPhone 13 over the course of two years, which could be attributed to various fact

In [25]:
ask_local_ai("Which month in 2024 had the cheapest average realistic price for the ipad? in which store and where the store")


💬  Which month in 2024 had the cheapest average realistic price for the ipad? in which store and where the store

⚙  SQL: SELECT city, store_name, AVG(price_realistic) AS avg_price_2024
FROM sales
WHERE product_name ILIKE 'ipad' AND year = 2024
GROUP BY city, store_name
ORDER BY avg_price_2024 ASC
LIMIT 1;

🔍  Raw Data: 

📊  Analyst Briefing:
   Based on the data provided, the cheapest average realistic price for an iPad in 2024 was $799. This occurred in the month of October at Apple's flagship store located at 1 Infinite Loop, Cupertino, CA.



In [26]:
# Specific Product Comparison by Year — Rule 12 ensures exact filtering
ask_local_ai("What is the average price of iPhone 13 and iPhone 14 for each year?")



💬  What is the average price of iPhone 13 and iPhone 14 for each year?

⚙  SQL: SELECT product_name, year, AVG(price_realistic) AS avg_price
FROM sales
WHERE (product_name ILIKE 'iPhone 13' OR product_name ILIKE 'iPhone 14')
GROUP BY product_name, year LIMIT 50;

🔍  Raw Data: [('iPhone 13', 2021, 791.6215596360211), ('iPhone 13', 2025, 630.8125269478052), ('iPhone 14', 2025, 632.9231111001288), ('iPhone 13', 2022, 756.0492524136782), ('iPhone 14', 2022, 806.2613300857581),...

📊  Analyst Briefing:
   **Executive Briefing**

The average price of iPhone 13 and iPhone 14 for each year is as follows:

- **iPhone 13**: 
  - 2021: $791.62
  - 2022: $756.05
  - 2023: $718.29
  - 2024: $646.73

- **iPhone 14**: 
  - 2022: $806.26
  - 2023: $781.28
  - 2024: $704.22

**Trend Analysis**: The average price of iPhone 14 has shown a slight increase compared to iPhone 13 over the years, with the highest prices occurring in 2022 and 2023.

**Actionable Takeaway**: Apple should consider adjusting pri

In [27]:
ask_local_ai("top 10 product for each category in 2023")


💬  top 10 product for each category in 2023

⚙  SQL: SELECT category_name, product_name, SUM(sales_amount_realistic) AS total_sales_2023
FROM sales
WHERE year = 2023
GROUP BY category_name, product_name
ORDER BY total_sales_2023 DESC
LIMIT 10;

🔍  Raw Data: [('Desktop', 'Mac Pro (M2 Ultra)', 85763313.3081173), ('Smartphone', 'iPhone 15 Pro', 84827230.15474069), ('Smartphone', 'iPhone 15 Plus', 83300734.87194407), ('Smartphone', 'iPhone 15', 79645210.8139...

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** Apple's top 10 products for each category in 2023 were:

- **Desktops:** Mac Pro (M2 Ultra) - Sales: $85.76 million
- **Smartphones:** iPhone 15 Pro, iPhone 15 Plus, iPhone 15, iPhone 15 Pro Max - Total Sales: $4,091.30 million
- **Wearables:** Apple Watch Ultra 2, Apple Watch Series 9 - Total Sales: $1,117.16 million
- **Laptops:** MacBook Pro 14/16-inch (M2/M3), MacBook Air 15-inch (M2) - Total Sales: $8,883.58 million

**Actionable Takeaways:**

1. **Smartphones D

In [28]:
ask_local_ai("What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.")


💬  What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.

⚙  SQL: SELECT 
    month, 
    SUM(sales_amount_realistic) AS total_revenue
FROM 
    sales
WHERE 
    country_norm_mapped = 'united states' AND 
    product_name ILIKE '%MacBook%' AND 
    year = 2024
GROUP BY 
    month
ORDER BY 
    month LIMIT 50;

🔍  Raw Data: [(1, 1581874.7483825993), (2, 1386672.0540408439), (3, 1695250.576629389), (4, 1526059.1524260303), (5, 1611689.0070899078), (6, 1848742.7134220887), (7, 1689644.9086284577), (8, 1494412.4531103794), ...

📊  Analyst Briefing:
   **Executive Briefing**

The total revenue generated for MacBooks in the United States from January to December 2024 is as follows:

- **January**: $1,581,874.75
- **February**: $1,386,672.05
- **March**: $1,695,250.58
- **April**: $1,526,059.15
- **May**: $1,611,689.01
- **June**: $1,848,742.71
- **July**: $1,689,644.91
- **August**: $1,494,412.45
- **September**: $

In [29]:
ask_local_ai("What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.")


💬  What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.

⚙  SQL: SELECT 
    month, 
    SUM(sales_amount_realistic) AS total_revenue
FROM 
    sales
WHERE 
    country_norm_mapped = 'united states' AND 
    product_name ILIKE '%MacBook%' AND 
    year = 2024
GROUP BY 
    month
ORDER BY 
    month LIMIT 50;

🔍  Raw Data: [(1, 1581874.7483825993), (2, 1386672.0540408439), (3, 1695250.576629389), (4, 1526059.1524260303), (5, 1611689.0070899078), (6, 1848742.7134220887), (7, 1689644.9086284577), (8, 1494412.4531103794), ...

📊  Analyst Briefing:
   **Executive Briefing**

The total revenue generated for MacBooks in the United States from January to December 2024 is as follows:

- **January**: $1,581,874.75
- **February**: $1,386,672.05
- **March**: $1,695,250.58
- **April**: $1,526,059.15
- **May**: $1,611,689.01
- **June**: $1,848,742.71
- **July**: $1,689,644.91
- **August**: $1,494,412.45
- **September**: $

In [30]:
ask_local_ai("what is the country that have the most sales of iphone 14?")


💬  what is the country that have the most sales of iphone 14?

⚙  SQL: SELECT country_norm_mapped, SUM(quantity_realistic) AS total_sales
FROM sales
WHERE product_name ILIKE 'iPhone 14'
GROUP BY country_norm_mapped
ORDER BY total_sales DESC
LIMIT 1;

🔍  Raw Data: [('united states', 18300)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** The United States has the highest sales of iPhone 14, with a total of 18,300 units sold.

**Supporting Details:** This data indicates that the US market was particularly strong for the iPhone 14 model. It suggests that the product resonated well with consumers in this region and contributed significantly to Apple's overall sales figures for the quarter or year.

**Business Context:** Understanding which countries have the most sales of a particular product is crucial for strategic planning, as it helps identify market trends and potential opportunities for expansion. This insight can guide Apple in tailoring marketing strategies and s

In [31]:
ask_local_ai("what is the average price of i phone 13 and i phone 14 for each year?")


💬  what is the average price of i phone 13 and i phone 14 for each year?

⚙  SQL: SELECT product_name, year, AVG(price_realistic) 
FROM sales 
WHERE (product_name ILIKE 'iPhone 13' OR product_name ILIKE 'iPhone 14') 
GROUP BY product_name, year LIMIT 50;

🔍  Raw Data: [('iPhone 14', 2022, 806.2613300857581), ('iPhone 13', 2022, 756.0492524136782), ('iPhone 14', 2024, 704.2181905774314), ('iPhone 13', 2024, 646.7346167649283), ('iPhone 14', 2023, 781.280033637729), ...

📊  Analyst Briefing:
   The average price of iPhone 13 and iPhone 14 for each year is as follows:

- **iPhone 13**: 
  - 2021: $791.62
  - 2022: $756.05
  - 2023: $718.29
  - 2024: $646.73
  - 2025: $630.81

- **iPhone 14**: 
  - 2021: N/A (no data available)
  - 2022: $806.26
  - 2023: $781.28
  - 2024: $704.22

**Actionable Takeaway**: The average price of iPhone 13 has been decreasing over the years, while the average price of iPhone 14 has remained relatively stable or slightly increased. This trend suggests that Ap

In [32]:
ask_local_ai("i phone 14 price in 2024 price by months sort it")


💬  i phone 14 price in 2024 price by months sort it

⚙  SQL: SELECT product_name, month, AVG(price_realistic) AS price_per_month
FROM sales
WHERE product_name ILIKE 'iPhone 14' AND year = 2024
GROUP BY product_name, month
ORDER BY month LIMIT 50;

🔍  Raw Data: [('iPhone 14', 1, 718.6179740640051), ('iPhone 14', 2, 699.7205135471255), ('iPhone 14', 3, 715.5817522308342), ('iPhone 14', 4, 700.5113187789692), ('iPhone 14', 5, 707.8463940171728), ('iPhone 14', ...

📊  Analyst Briefing:
   The iPhone 14 price trend in 2024 shows a steady increase throughout the year. The lowest price recorded was $690.95 on November, while the highest was $718.62 on January. This indicates that the price of the iPhone 14 has been rising steadily over the course of the year, with no significant fluctuations observed.



In [33]:
ask_local_ai("iphone 13 price in 2024 price by months sort it show to me by months sort them")




💬  iphone 13 price in 2024 price by months sort it show to me by months sort them

⚙  SQL: SELECT city, month, AVG(price_realistic) AS price_2024 
FROM sales 
WHERE product_name ILIKE 'iPhone 13' AND year = 2024 
GROUP BY city, month 
ORDER BY month LIMIT 50;

🔍  Raw Data: [('Cheltenham', 1, 684.6147690604622), ('Munich', 1, 740.3912240780795), ('Portland', 1, 646.7237571697972), ('Bangkok', 1, 682.8519804182866), ('Paris', 1, 666.2411422686221), ('Burnaby', 1, 647.4654...

📊  Analyst Briefing:
   **Executive Briefing**

Good morning, Apple executives.

Today, I'm presenting findings on the iPhone 13 price trends in various cities for the month of January and February 2024. The data shows a clear upward trend in prices across all cities, with the highest average price being $740.39 in Munich.

Here's a breakdown by city:

- **Munich**: $740.39
- **Cheltenham**: $684.61
- **Portland**: $646.72
- **Bangkok**: $682.85
- **Paris**: $666.24
- **Burnaby**: $647.47
- **Vienna**: $673.89
- **